In [1]:
import pandas as pd

# EXTRACT: load the machine-readable CSV (all tables pre-combined)
df = pd.read_csv(r"C:\Users\manoj\OneDrive\Desktop\Project 2\ETL walkthrough\rogs-2026-parte-section12-public-hospitals-dataset_0.csv")

print("Shape:", df.shape)
df.head()

Shape: (9575, 30)


,Table_Number,Year,Measure,Age,Sex,Indigenous_Status,Remoteness,SEIFA_IRSD,Hospital_Sector_Grp,Triage_Category,...,NSW,Vic,Qld,WA,SA,Tas,ACT,NT,Total,Aust
0,12A.1,2023-24,Recurrent expenditure,All ages,All people,All people,All areas,NaN,Public hospitals (including psychiatric hospit...,NaN,...,15995,16399,14195,6656,3334,1746,1214,1084,NaN,60623
1,12A.1,2023-24,Recurrent expenditure,All ages,All people,All people,All areas,NaN,Public hospitals (including psychiatric hospit...,NaN,...,13700,9287,7555,5757,3330,1075,948,776,NaN,42428
2,12A.1,2023-24,Recurrent expenditure,All ages,All people,All people,All areas,NaN,Public hospitals (including psychiatric hospit...,NaN,...,29696,25686,21750,12414,6664,2821,2162,1860,NaN,103051
3,12A.1,2022-23,Recurrent expenditure,All ages,All people,All people,All areas,NaN,Public hospitals (including psychiatric hospit...,NaN,...,15862,15981,14155,6607,3386,1551,1156,841,NaN,59539
4,12A.1,2022-23,Recurrent expenditure,All ages,All people,All people,All areas,NaN,Public hospitals (including psychiatric hospit...,NaN,...,13775,9163,6960,5830,3265,1012,872,589,NaN,41466


In [2]:
#inspect column names
print(df.columns.tolist())

['Table_Number', 'Year', 'Measure', 'Age', 'Sex', 'Indigenous_Status', 'Remoteness', 'SEIFA_IRSD', 'Hospital_Sector_Grp', 'Triage_Category', 'Year_Dollars', 'Description1', 'Description2', 'Description3', 'Description4', 'Description5', 'Description6', 'Uncertainty', 'Data_Source', 'Unit', 'NSW', 'Vic', 'Qld', 'WA', 'SA', 'Tas', 'ACT', 'NT', 'Total', 'Aust']


In [3]:
#data Info
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9575 entries, 0 to 9574
Data columns (total 30 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   Table_Number         9575 non-null   object
 1   Year                 9575 non-null   object
 2   Measure              9575 non-null   object
 3   Age                  9575 non-null   object
 4   Sex                  9575 non-null   object
 5   Indigenous_Status    9575 non-null   object
 6   Remoteness           9575 non-null   object
 7   SEIFA_IRSD           1065 non-null   object
 8   Hospital_Sector_Grp  6897 non-null   object
 9   Triage_Category      2570 non-null   object
 10  Year_Dollars         216 non-null    object
 11  Description1         9380 non-null   object
 12  Description2         8494 non-null   object
 13  Description3         7730 non-null   object
 14  Description4         1881 non-null   object
 15  Description5         5811 non-null   object
 16  Descri

In [4]:
#Measures
print(df['Measure'].unique())

['Recurrent expenditure' 'Hospital size' 'Available beds'
 'Summary of separations' 'Acute same-day and overnight separations'
 'Workforce' 'Individual and group service events'
 'Emergency department presentations' 'Emergency department waiting times'
 'Patients staying for four hours or less'
 'Elective surgery: waiting times'
 'Elective surgery waiting list turn over'
 'Discharge against medical advice' 'Public hospital accreditation'
 'Selected healthcare-associated infections'
 'Adverse events treated in hospitals'
 'Falls resulting in patient harm in hospitals'
 'Sentinel events; public hospitals' 'Patient satisfaction'
 'Unplanned hospital readmissions' 'Workforce sustainability'
 'Recurrent cost per weighted separation'
 'Capital cost per weighted separation'
 'Recurrent cost per non-admitted patient']


In [5]:
#Data Transformation
id_columns = ['Table_Number', 'Year', 'Measure', 'Description2', 
              'Description3', 'Unit']

state_columns = ['NSW', 'Vic', 'Qld', 'WA', 'SA', 'Tas', 'ACT', 'NT']

# melt() reshapes wide -> long
df_long = df.melt(
    id_vars=id_columns,        # columns to keep as-is
    value_vars=state_columns,  # columns to unpivot
    var_name='State',          # new column holding the state name
    value_name='Value'         # new column holding the number
)


In [6]:
print("Before:", df.shape)
print("After: ", df_long.shape)
df_long.head(10)

Before: (9575, 30)
After:  (76600, 8)


,Table_Number,Year,Measure,Description2,Description3,Unit,State,Value
0,12A.1,2023-24,Recurrent expenditure,Real recurrent expenditure,Salary and wages,$m,NSW,15995
1,12A.1,2023-24,Recurrent expenditure,Real recurrent expenditure,Non-salary,$m,NSW,13700
2,12A.1,2023-24,Recurrent expenditure,Real recurrent expenditure,Total,$m,NSW,29696
3,12A.1,2022-23,Recurrent expenditure,Real recurrent expenditure,Salary and wages,$m,NSW,15862
4,12A.1,2022-23,Recurrent expenditure,Real recurrent expenditure,Non-salary,$m,NSW,13775
5,12A.1,2022-23,Recurrent expenditure,Real recurrent expenditure,Total,$m,NSW,29637
6,12A.1,2021-22,Recurrent expenditure,Real recurrent expenditure,Salary and wages,$m,NSW,15905
7,12A.1,2021-22,Recurrent expenditure,Real recurrent expenditure,Non-salary,$m,NSW,14661
8,12A.1,2021-22,Recurrent expenditure,Real recurrent expenditure,Total,$m,NSW,30566
9,12A.1,2020-21,Recurrent expenditure,Real recurrent expenditure,Salary and wages,$m,NSW,14848


In [7]:
# Check the data type of Value — it should say 'object' (text)
print("Value column type:", df_long['Value'].dtype)

# Find the non-numeric values hiding in there
# pd.to_numeric with errors='coerce' turns numbers into numbers, junk into NaN
numeric_test = pd.to_numeric(df_long['Value'], errors='coerce')

# Rows where conversion failed but the original wasn't already empty
problem_values = df_long['Value'][numeric_test.isna() & df_long['Value'].notna()]

print("\nNon-numeric values found:")
print(problem_values.unique())

Value column type: object

Non-numeric values found:
['na' 'np' '..']


In [8]:
#Data Transformation continuation
import numpy as np

suppressed_tokens = ['na','np' '..']

# Replace each token with NaN (proper "missing" marker)
df_long['Value'] = df_long['Value'].replace(suppressed_tokens, np.nan)

df_long['Value'] = pd.to_numeric(df_long['Value'], errors='coerce')

print("Value column type now:", df_long['Value'].dtype)
print("Missing values:", df_long['Value'].isna().sum())
df_long.head()


Value column type now: float64
Missing values: 20550


,Table_Number,Year,Measure,Description2,Description3,Unit,State,Value
0,12A.1,2023-24,Recurrent expenditure,Real recurrent expenditure,Salary and wages,$m,NSW,15995.0
1,12A.1,2023-24,Recurrent expenditure,Real recurrent expenditure,Non-salary,$m,NSW,13700.0
2,12A.1,2023-24,Recurrent expenditure,Real recurrent expenditure,Total,$m,NSW,29696.0
3,12A.1,2022-23,Recurrent expenditure,Real recurrent expenditure,Salary and wages,$m,NSW,15862.0
4,12A.1,2022-23,Recurrent expenditure,Real recurrent expenditure,Non-salary,$m,NSW,13775.0


In [9]:
# TRANSFORM Step 3: drop rows with no usable value
rows_before = len(df_long)

df_clean = df_long.dropna(subset=['Value']).reset_index(drop=True)

rows_after = len(df_clean)
print(f"Dropped {rows_before - rows_after} rows with missing values")
print(f"Clean dataset: {rows_after} rows")

Dropped 20550 rows with missing values
Clean dataset: 56050 rows


In [10]:
# TRANSFORM Step 4: map state abbreviations to full names
state_mapping = {
    'NSW': 'New South Wales',
    'Vic': 'Victoria',
    'Qld': 'Queensland',
    'WA': 'Western Australia',
    'SA': 'South Australia',
    'Tas': 'Tasmania',
    'ACT': 'Australian Capital Territory',
    'NT': 'Northern Territory'
}

# .map() looks up each abbreviation and returns the full name
df_clean['StateName'] = df_clean['State'].map(state_mapping)

df_clean.head()

,Table_Number,Year,Measure,Description2,Description3,Unit,State,Value,StateName
0,12A.1,2023-24,Recurrent expenditure,Real recurrent expenditure,Salary and wages,$m,NSW,15995.0,New South Wales
1,12A.1,2023-24,Recurrent expenditure,Real recurrent expenditure,Non-salary,$m,NSW,13700.0,New South Wales
2,12A.1,2023-24,Recurrent expenditure,Real recurrent expenditure,Total,$m,NSW,29696.0,New South Wales
3,12A.1,2022-23,Recurrent expenditure,Real recurrent expenditure,Salary and wages,$m,NSW,15862.0,New South Wales
4,12A.1,2022-23,Recurrent expenditure,Real recurrent expenditure,Non-salary,$m,NSW,13775.0,New South Wales


In [11]:
print("Final clean dataset shape:", df_clean.shape)

Final clean dataset shape: (56050, 9)


In [12]:
# LOAD: write the clean data into a SQLite database
import sqlite3

# Connect to a database file (creates it if it doesn't exist)
conn = sqlite3.connect('hospital_analytics.db')

# Write the DataFrame to a table called 'fact_hospital_performance'
df_clean.to_sql(
    'fact_hospital_performance',  # table name
    conn,                          # the database connection
    if_exists='replace',           # overwrite if it already exists
    index=False                    # don't save the pandas row numbers
)

print("Data loaded into database!")

Data loaded into database!


In [13]:
# VERIFY: query the database with SQL to confirm the load worked
query = """
SELECT StateName, SUM(Value) AS total_presentations
FROM fact_hospital_performance
WHERE Measure = 'Emergency department presentations'
  AND Description3 = 'Presentations'
  AND Unit = 'no.'
  AND Year = '2024-25'
GROUP BY StateName
ORDER BY total_presentations DESC
"""

result = pd.read_sql_query(query, conn)
result

,StateName,total_presentations
0,New South Wales,6320946.0
1,Victoria,3956534.0
2,Queensland,3533678.0
3,Western Australia,2054034.0
4,South Australia,1245752.0
5,Northern Territory,376234.0
6,Tasmania,366240.0
7,Australian Capital Territory,335206.0


In [14]:
# Australian Healthcare ETL Pipeline

#This notebook extracts, transforms, and loads Australian public hospital 
#performance data (ROGS 2026) into a SQLite database.

#1. Extract — load the machine-readable CSV
#2. Transform — unpivot states to long format, clean suppressed values, type-cast
#3. Load — write to SQLite
#4. Verify — aggregate SQL queries

#**Key decisions:**
#- Used the machine-readable CSV over the 59-sheet Excel workbook for reproducibility
#- Converted suppression codes (np/na/..) to NaN rather than zero, since 
#  suppressed values are unknown, not zero
#- Dropped null-value rows for a clean database

In [15]:
conn.close()
print("Pipeline complete. Database connection closed.")

Pipeline complete. Database connection closed.


In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect('hospital_analytics.db')
print("Connected to database")

Connected to database


In [2]:
query = """
SELECT 
    Measure,
    COUNT(*) AS record_count
FROM fact_hospital_performance
GROUP BY Measure
ORDER BY record_count DESC
"""

pd.read_sql_query(query, conn)

,Measure,record_count
0,Elective surgery: waiting times,14643
1,Emergency department waiting times,10211
2,Workforce sustainability,8060
3,Patient satisfaction,4536
4,Patients staying for four hours or less,3104
5,Summary of separations,2074
6,Adverse events treated in hospitals,1760
7,Emergency department presentations,1618
8,Acute same-day and overnight separations,1240
9,Falls resulting in patient harm in hospitals,1071


In [3]:
query = """
SELECT 
    Year,
    ROUND(AVG(Value), 1) AS avg_pct_within_4hrs
FROM fact_hospital_performance
WHERE Measure = 'Patients staying for four hours or less'
  AND Description3 = 'ED stay length is within four hours'
  AND Unit = '%'
GROUP BY Year
ORDER BY Year
"""

pd.read_sql_query(query, conn)

,Year,avg_pct_within_4hrs
0,2015-16,70.4
1,2016-17,69.8
2,2017-18,67.8
3,2018-19,66.5
4,2019-20,66.2
5,2020-21,63.9
6,2021-22,58.7
7,2022-23,54.6
8,2023-24,54.3
9,2024-25,53.0


In [4]:
query = """
SELECT 
    Year,
    ROUND(AVG(Value), 1) AS avg_pct,
    CASE 
        WHEN AVG(Value) >= 70 THEN 'On Target'
        WHEN AVG(Value) >= 60 THEN 'Concern'
        ELSE 'Critical'
    END AS performance_status
FROM fact_hospital_performance
WHERE Measure = 'Patients staying for four hours or less'
  AND Description3 = 'ED stay length is within four hours'
  AND Unit = '%'
GROUP BY Year
ORDER BY Year
"""

pd.read_sql_query(query, conn)

,Year,avg_pct,performance_status
0,2015-16,70.4,On Target
1,2016-17,69.8,Concern
2,2017-18,67.8,Concern
3,2018-19,66.5,Concern
4,2019-20,66.2,Concern
5,2020-21,63.9,Concern
6,2021-22,58.7,Critical
7,2022-23,54.6,Critical
8,2023-24,54.3,Critical
9,2024-25,53.0,Critical


In [5]:
query = """
SELECT 
    Year,
    ROUND(AVG(Value), 1) AS avg_pct,
    ROUND(
        AVG(Value) - LAG(AVG(Value)) OVER (ORDER BY Year),
        1
    ) AS change_from_prev_year
FROM fact_hospital_performance
WHERE Measure = 'Patients staying for four hours or less'
  AND Description3 = 'ED stay length is within four hours'
  AND Unit = '%'
GROUP BY Year
ORDER BY Year
"""

pd.read_sql_query(query, conn)

,Year,avg_pct,change_from_prev_year
0,2015-16,70.4,NaN
1,2016-17,69.8,-0.7
2,2017-18,67.8,-2.0
3,2018-19,66.5,-1.3
4,2019-20,66.2,-0.3
5,2020-21,63.9,-2.3
6,2021-22,58.7,-5.2
7,2022-23,54.6,-4.1
8,2023-24,54.3,-0.3
9,2024-25,53.0,-1.3


In [6]:
query = """
SELECT 
    StateName,
    ROUND(AVG(Value), 1) AS avg_pct,
    RANK() OVER (ORDER BY AVG(Value) DESC) AS performance_rank
FROM fact_hospital_performance
WHERE Measure = 'Patients staying for four hours or less'
  AND Description3 = 'ED stay length is within four hours'
  AND Unit = '%'
  AND Year = '2024-25'
  AND State NOT IN ('Total', 'Aust')
GROUP BY StateName
ORDER BY performance_rank
"""

pd.read_sql_query(query, conn)

,StateName,avg_pct,performance_rank
0,Australian Capital Territory,60.8,1
1,New South Wales,56.3,2
2,Western Australia,53.9,3
3,Northern Territory,53.6,4
4,Victoria,52.7,5
5,Queensland,49.9,6
6,South Australia,49.6,7
7,Tasmania,47.5,8


In [7]:
query = """
WITH state_averages AS (
    SELECT 
        StateName,
        AVG(Value) AS state_avg
    FROM fact_hospital_performance
    WHERE Measure = 'Patients staying for four hours or less'
      AND Description3 = 'ED stay length is within four hours'
      AND Unit = '%'
      AND Year = '2024-25'
      AND State NOT IN ('Total', 'Aust')
    GROUP BY StateName
),
national AS (
    SELECT AVG(state_avg) AS national_avg
    FROM state_averages
)
SELECT 
    s.StateName,
    ROUND(s.state_avg, 1) AS state_avg,
    ROUND(n.national_avg, 1) AS national_avg,
    ROUND(s.state_avg - n.national_avg, 1) AS gap
FROM state_averages s
CROSS JOIN national n
WHERE s.state_avg < n.national_avg
ORDER BY gap
"""

pd.read_sql_query(query, conn)

,StateName,state_avg,national_avg,gap
0,Tasmania,47.5,53.0,-5.5
1,South Australia,49.6,53.0,-3.4
2,Queensland,49.9,53.0,-3.1
3,Victoria,52.7,53.0,-0.3
